# Week 56, Day 2: Timed set 3

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w056d2.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


*Interactive figure on the website: The timer keeps running if you reload. It doesn't stop you working when time is up..*

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: P1: An order for the course

Courses are numbered 0 to n − 1, and `prerequisites` are pairs `(a, b)` meaning a must be taken before b. Write `course_order(n, prerequisites)`: an order taking every course after its prerequisites, choosing the **lowest-numbered** available course at each step, or `None` if no order exists.

In [ ]:
import heapq

def course_order(n, prerequisites):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import itertools, random\ndef test_order():\n    \"topological order\"\n    assert course_order(4, [(1, 0), (2, 0), (3, 1), (3, 2)]) == [3, 1, 2, 0]\n    assert course_order(3, []) == [0, 1, 2] and course_order(2, [(0, 1), (1, 0)]) is None and course_order(0, []) == []\n    rng = random.Random(1)\n    for _ in range(100):\n        n = rng.randint(1, 7)\n        pre = list({(rng.randrange(n), rng.randrange(n)) for _ in range(rng.randint(0, 8))})\n        valid = [p for p in itertools.permutations(range(n)) if all(p.index(a) < p.index(b) for a, b in pre)]\n        got = course_order(n, pre)\n        assert got == (list(min(valid)) if valid else None), (n, pre)")

### Exercise: P2: How long until everyone hears?

A message starts at node `start` of a network of nodes 1 to n. `links` are `(u, v, t)`: a message at u reaches v after t milliseconds (one way; t ≥ 0). Write `time_to_reach_all(n, links, start)`: when the last node hears the message, or `-1` if some node never does. Then `last_to_hear(n, links, start)`: the sorted list of nodes that hear it last (`[]` if some node never does).

In [ ]:
import heapq

def time_to_reach_all(n, links, start):
    ___

def last_to_hear(n, links, start):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_delay():\n    \"Dijkstra\"\n    links = [(2, 1, 1), (2, 3, 1), (3, 4, 1)]\n    assert time_to_reach_all(4, links, 2) == 2 and last_to_hear(4, links, 2) == [4]\n    assert time_to_reach_all(2, [(1, 2, 1)], 2) == -1 and last_to_hear(2, [(1, 2, 1)], 2) == []\n    assert time_to_reach_all(1, [], 1) == 0 and last_to_hear(1, [], 1) == [1]\n    rng = random.Random(2)\n    for _ in range(100):\n        n = rng.randint(1, 7)\n        links = [(rng.randint(1, n), rng.randint(1, n), rng.randint(0, 9)) for _ in range(rng.randint(0, 15))]\n        s = rng.randint(1, n)\n        dist = {v: float(\"inf\") for v in range(1, n + 1)}\n        dist[s] = 0\n        for _ in range(n):\n            for u, v, t in links:\n                dist[v] = min(dist[v], dist[u] + t)\n        worst = max(dist.values())\n        assert time_to_reach_all(n, links, s) == (-1 if worst == float(\"inf\") else worst)\n        if worst < float(\"inf\"):\n            assert last_to_hear(n, links, s) == sorted(v for v in dist if dist[v] == worst)")

### Exercise: P3: Bursting balloons with arrows

Balloons are horizontal intervals `(start, end)` (inclusive). An arrow shot straight up at x bursts every balloon with start ≤ x ≤ end. Write `fewest_arrows(balloons)`: the fewest arrows to burst them all, and `arrow_positions(balloons)`: a sorted list of positions for such a set of arrows (each arrow at the **end** of the balloon that forced it).

In [ ]:
def fewest_arrows(balloons):
    ___

def arrow_positions(balloons):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import itertools, random\ndef test_arrows():\n    \"stabbing intervals\"\n    b = [(10, 16), (2, 8), (1, 6), (7, 12)]\n    assert fewest_arrows(b) == 2 and arrow_positions(b) == [6, 12]\n    assert fewest_arrows([]) == 0 and fewest_arrows([(1, 2), (3, 4), (5, 6), (7, 8)]) == 4 and fewest_arrows([(1, 2), (2, 3)]) == 1\n    rng = random.Random(3)\n    for _ in range(150):\n        bs = [(s, s + rng.randint(0, 6)) for s in (rng.randint(0, 20) for _ in range(rng.randint(1, 7)))]\n        pos = arrow_positions(bs)\n        assert all(any(s <= x <= e for x in pos) for s, e in bs) and pos == sorted(pos)\n        points = sorted({p for s, e in bs for p in (s, e)})\n        best = next(k for k in range(1, len(bs) + 1)\n                    if any(all(any(s <= x <= e for x in c) for s, e in bs) for c in itertools.combinations(points, k)))\n        assert fewest_arrows(bs) == best == len(pos)")

### Exercise: P4: Cutting a log

A log of length L must be cut at given positions (whole numbers between 0 and L). Each cut costs the length of the piece being cut, and you may do the cuts in any order. Write `cheapest_cuts(L, cuts)`: the least total cost, and `cut_order(L, cuts)`: one cheapest order of cut positions (at each stretch, cut first at the best split point, the leftmost on a tie, then do its left part's cuts before its right part's).

In [ ]:
def cheapest_cuts(L, cuts):
    ___

def cut_order(L, cuts):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import itertools, random\ndef cost_of(L, order):\n    pieces, total = [(0, L)], 0\n    for c in order:\n        for i, (a, b) in enumerate(pieces):\n            if a < c < b:\n                total += b - a\n                pieces[i:i + 1] = [(a, c), (c, b)]\n                break\n    return total\ndef test_cuts():\n    \"interval DP\"\n    assert cheapest_cuts(7, [1, 3, 4, 5]) == 16 and cheapest_cuts(9, [5, 6, 1, 4, 2]) == 22\n    assert cheapest_cuts(10, []) == 0 and cheapest_cuts(10, [5]) == 10 and cut_order(10, [5]) == [5]\n    order = cut_order(7, [1, 3, 4, 5])\n    assert sorted(order) == [1, 3, 4, 5] and cost_of(7, order) == 16\n    rng = random.Random(4)\n    for _ in range(60):\n        L = rng.randint(2, 20)\n        cuts = rng.sample(range(1, L), rng.randint(0, min(5, L - 1)))\n        best = min(cost_of(L, p) for p in itertools.permutations(cuts)) if cuts else 0\n        assert cheapest_cuts(L, cuts) == best and cost_of(L, cut_order(L, cuts)) == best\n    big = rng.sample(range(1, 10 ** 6), 100)\n    assert cheapest_cuts(10 ** 6, big) > 0")

### Exercise: P5: No two ones together

Write `count_no_adjacent(n)`: how many whole numbers from 0 to n (inclusive) have **no two neighbouring 1 bits** in binary (5 = `101` does; 6 = `110` doesn't), for n up to 10¹⁸. A digit DP over the bits of n, from the top, with the state (position, previous bit, tight). Then `nth_no_adjacent(k)`: the k-th such number counting from 0 (so `nth_no_adjacent(0) == 0`), by binary search on `count_no_adjacent`.

In [ ]:
from functools import cache

def count_no_adjacent(n):
    ___

def nth_no_adjacent(k):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_bits():\n    \"digit DP on bits\"\n    good = [x for x in range(5000) if x & (x >> 1) == 0]\n    assert [count_no_adjacent(n) for n in range(10)] == [1, 2, 3, 3, 4, 5, 5, 5, 6, 7]\n    for n in range(0, 5000, 7):\n        assert count_no_adjacent(n) == sum(1 for x in good if x <= n)\n    assert [nth_no_adjacent(k) for k in range(8)] == good[:8]\n    assert all(nth_no_adjacent(k) == good[k] for k in range(0, len(good), 13))\n    assert count_no_adjacent(2 ** 60 - 1) == 4052739537881 and nth_no_adjacent(10 ** 9) & (nth_no_adjacent(10 ** 9) >> 1) == 0")

## How to use this

Treat the next 100 minutes as the real thing: notes closed, no tutor, no web, timer on. Run your code and press **Check** as often as you like. **Don't open the hints or solutions until the timer has finished**; then go through every problem and compare.

Start every answer with a comment: `# pattern: ..., cost: ...`.

**The marks.**

| Part | What | Marks |
|---|---|---|
| A | 10 quiz questions | 10 (1 each) |
| B | 5 problems | 80 |
| | **Total** | **90** |

Count full marks for a problem only when every check passes. 54 (60%) is a good result; 72 (80%) an excellent one.

## Part A: quiz

## Part B: five problems

## Afterwards

When the timer ends, add up your marks, then go back through each problem with its hint and solution. For every mark you dropped, decide which kind of reason it was: **didn't know it**, **picked the wrong pattern**, **an edge case**, or **ran out of time**. Each needs a different fix (revise, note the signal you missed, test small cases first, or pace better).

Which patterns did today's problems need?

- P1: Kahn's topological sort (Week 50), with a heap for the smallest available course (Week 47).
- P2: Dijkstra (Week 51), then the largest distance.
- P3: greedy by end time, the interval stabbing argument (Week 52 Day 2).
- P4: interval DP over the cut points (Week 54 Day 5), the same shape as merging piles and matrix chains.
- P5: digit DP (Week 54 Day 5) on binary digits (Week 55), then binary search on the answer (Week 18 and Monday).

Tomorrow: timed set 4, with no hints and no solutions until Saturday's debrief.